# Pareto smoothed importance sampling leave-one-out (PSIS LOO)

### Background and Motivation

We often need to evaluate the performance of the Bayesian models on a test data or out of sample data after training (MCMC) for model comparison or selection. Crossvalidation (CV) is one of the effective methods for estimating the out of sample predictive accuracy.The predicitive measures explored so far AIC, DIC and WAIC; all can be viewed as approximations to different versions of cross validation. 

As we learn from the lectures, AIC is a frequentist criterion and DIC, often called the bayesian counterpart of AIC, is somewhat bayesian. DIC is not fully bayesian because it directly plugs in the posterior mean of the parameters $\bar{\theta}$ into the log likelihood. Whereas, WAIC and LOO CV are fully bayesian in the sense that they integrate over the full posteior instead of relying on a point estimate.  

The modern metric WAIC, although asymptotically equivalent to LOO CV, has been shown in [Vehtari, Gelman, and Gabry (2017)](https://arxiv.org/abs/1507.04544) to be less robust in cases with weak priors and influential outliers. In many examples tested in the above paper, PSIS LOO CV also showed smaller error and lower bias than WAIC. Therefore, due to its robustness and reliability, PSIS LOO CV is widely preferred over WAIC in modern PPL. WAIC has even been [removed](https://python.arviz.org/en/latest/user_guide/migration_guide.html#model-comparison) from the latest version of ArviZ.

Now lets dive into PSIS-LOO.

### LOO CV 
Exact leave one out cross validation (LOO CV) requires performing MCMC on several training datasets and testing on the one left out datapoint each time. The computation can be very time consuming (from a few days to several months), depending on the size of the size of the dataset. However, an approximation of leave one out cross validation (LOO CV) can be estimated using importance sampling within minutes.

Consider data points $y_1,\ldots,y_n$ which are independent given parameters $\theta$;

Likelihood: $p(y|\theta)=\prod_{i=1}^{n} p(y_i|\theta)$

Assume prior: $p(\theta)$

Yielding posterior: $p(\theta|y)$

The Bayesian LOO CV estimate is represented using (elpd) expected log pointwise predictive desnsity:

$$
\text{elpd}_{\text{loo-cv}}
=
\sum_{i=1}^{n}
\log p(y_i|y_{-i}),
$$
where

$$
p(y_i|y_{-i})
=
\int p(y_i|\theta)p(\theta|y_{-i})d\theta
$$

is the leave-one-out predictive density given the data without the ith data point.

From [Gelfand, Dey, and Chang (1992)](https://statistics.stanford.edu/technical-reports/model-determination-using-predictive-distributions-implementation-sampling-based), the exact loo estimate; $p(y_i|y_{-i})$ can be approximated by importance sampling.  The approximation is obtained using posteior draws $\theta^s$ from the full posterior distribution $p(\theta|y)$ and corresponding importance ratios.

### Importance ratios
Assume the number of posterior samples is *S*. Importance ratio for observation *i* with posterior sample index *s* is given by: 
$$
r_i^s
=
\frac{1}{p(y_i|\theta^s)}
\propto
\frac{p(\theta^s|y_{-i})}{p(\theta^s|y)}
$$
  
Importance sampling leave-one-out (IS-LOO) predictive distribution:

$$
p(\tilde{y}_i|y_{-i})
\approx
\frac{\sum_{s=1}^{S} r_i^s p(\tilde{y}_i|\theta^s)}
{\sum_{s=1}^{S} r_i^s}.
$$

Evaluating at the held-out data point $y_i$, we get

$$
p(y_i|y_{-i})
\approx
\frac{1}
{\frac{1}{S}\sum_{s=1}^{S}\frac{1}{p(y_i|\theta^s)}}.
$$

The problem with IS-LOO is that the posterior $p(\theta|y)$ is likely to have a smaller variance and thinner tails than $p(\theta|y_{-i})$. Thus, a direct use of the above expression for $p(y_i|y_{-i})$ induces instability because the importance ratios/weights can have large or infinite variance.

### Pareto smoothed importance sampling (PSIS)
The distribution of the importance weights used in LOO may have a long right tail, so direct sampling can lead to one or a few very large weights. PSIS applies a smoothing procedure to the importance weights by fitting a Pareto distribution to the upper tail of the importance ratios ($r_i^s$). The largest weights in $r_i^s$ are then replaced by expected quantiles $w_i^s$ from the fitted Pareto distribution, which are more well behaved than the original $r_i^s$ from which they are constructed.

#### PSIS LOO CV

PSIS estimate of the LOO expected log pointwise predictive density is given by:
$$
\text{elpd}_{\text{psis-loo-cv}}
=
\sum_{i=1}^{n}
\log
\left(
\frac{
\sum_{s=1}^{S} w_i^s \, p(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} w_i^s
}
\right).
$$

The shape parameter *k* of the modeled Pareto distribution can be used to assess the reliability of the estimate:

| Pareto shape parameter | Interpretation |
|---|---|
| *k* $< 1 - \frac{1}{\log_{10}(S)}$ | The PSIS estimate is expected to be accurate. |
| *k* $< \min\left(1 - \frac{1}{\log_{10}(S)},\, 0.7\right)$ | The PSIS estimate is expected to be reliable. |
| $0.7 < $ *k* $ < 1$ | It becomes computationally expensive to obtain an accurate estimate. |
| *k* $> 1$ | Mean and variance of the importance weights does not exist, and PSIS estimates become invalid. |
### PSIS LOO PIT 
The ordinary leave-one-out (LOO) probability integral transform (PIT) value for observation $y_i$ is

$$
\text{PIT}_i
=
P(\tilde y_i \le y_i \mid y_{-i})
=
\int
P(\tilde y_i \le y_i \mid \theta)
\, p(\theta \mid y_{-i})
\, d\theta.
$$

Using importance sampling:

$$
\text{PIT}_i
\approx
\frac{
\sum_{s=1}^{S}
r_i^s \,
F(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} r_i^s
},
$$

where

$$
F(y_i \mid \theta^s)
=
P(\tilde y_i \le y_i \mid \theta^s)
$$

is the posterior predictive cumulative distribution function (CDF) evaluated at the observed value $y_i$.

In Pareto smoothed importance sampling leave-one-out (PSIS-LOO), the unstable importance ratios $r_i^s$ are replaced by Pareto-smoothed weights $w_i^s$:

$$
\text{PSIS-LOO-PIT}_i
\approx
\frac{
\sum_{s=1}^{S}
w_i^s \,
F(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} w_i^s
}.
$$

### When to use
PSIS-LOO-CV is primarily used for model comparison and model selection. When we have two or more candidate models, elpd values can be used to evaluate which model is expected to generalize better. A higher elpd value indicates that a model is expected to predict unseen data more accurately. 

PSIS-LOO-PIT calculates the probability integral transform (CDF) and is used to evaluate the reliability of the model. If the PIT values $\text{PSIS-LOO-PIT}_i$ follow a uniform distribution, then the Bayesian model is considered reliable.

Both metrics can also be used to detect outliers. In PSIS-LOO-CV, if a data point has a highly negative $\text{elpd}_{i}$ value, then it is a potential outlier. Similarly, in PSIS-LOO-PIT, if a data point has PIT values $\text{PSIS-LOO-PIT}_i$ close to 0 or 1, then it lies in the long tail of the predictive distribution, again indicating that the point may be an outlier.

### Example
In the below example, we evaluate the [Eight School Problem](https://www.jstor.org/stable/1164617) using PSIS LOO.  

Eight different high schools each tried a special SAT coaching program to improve student test scores. Each school ran its own experiment and reported how much improvement they observed. 
Some schools reported large improvements, some reported small improvements, and a few even reported negative effects. Since the 8 SAT programs are not completely unrelated. The question is, if one school reports a very good improvement of student scores, is that school truly special, or did it just get lucky because of random variation?

In [9]:
import arviz as az
import numpy as np
import pymc as pm
import arviz_stats
import pandas as pd

# coaching effect estimates
y = [28,  8, -3,  7, -1,  1, 18, 12] 

# standard error in the above estimates
σ = [15, 10, 16, 11,  9, 11, 10, 18]

### Hierarchical model for the Eight Schools problem

$$
\begin{align*}
y_{j} & \sim \mathcal{N}(\theta_j, \sigma_j) && \text{likelihood}, \space j = 1 , ... , 8 \\
\theta_j & \sim \mathcal{N}(\mu, \tau)\\
\mu & \sim \mathcal{N}(0, 100)  && \text{prior: } \mu\\
\tau & \sim \text{Half-Normal}(25) && \text{prior: } \tau
\end{align*}
$$

where,

$y_j$: observed coaching effect estimate for school j

$\theta_j$: true (unknown) coaching treatment effect for school j

$\sigma_j$: standard error in the observed coaching effect of school j

$\mu$: overall mean treatment effect across all schools

$\tau$: standard deviation representing variability of true effects between schools

In [14]:
with pm.Model() as m:
    # μ = pm.Normal("μ", mu=0, sigma=100)
    # τ = pm.HalfNormal("τ", sigma=25)
    μ = pm.Normal("μ", mu=0, sigma=50)
    τ = pm.HalfNormal("τ", sigma=10)
    θ = pm.Normal("theta",mu=μ,sigma=τ, shape=len(y))
    pm.Normal("likelihood",mu=θ,sigma=σ, observed=y)
    trace = pm.sample(draws=3000,tune=2000,target_accept=0.95, 
        return_inferencedata=True)
    pm.compute_log_likelihood(trace)
    pm.sample_posterior_predictive(trace, extend_inferencedata=True)

c:\ProgramData\miniconda3\envs\pymc_env\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
NUTS[nutpie]: [μ, τ, theta]


Output()

Output()

Sampling: [likelihood]


Output()

In [ ]:
loo_pit_output = arviz_stats.loo_pit(trace, var_names="likelihood")
loo_cv = arviz_stats.loo(trace, var_name="likelihood", pointwise=True)

df = pd.DataFrame({"y": [28,  8, -3,  7, -1,  1, 18, 12], "σ": [15, 10, 16, 11,  9, 11, 10, 18]})
df["loo_pit"] = loo_pit_output["likelihood"].values
df["elpd_loo_cv"] =loo_cv.elpd_i.values
df["pareto_k"] = loo_cv.pareto_k.values

df

,y,σ,loo_pit,elpd_loo_cv,pareto_k
0,28,15,0.906424,-4.603149,0.510458
1,8,10,0.499927,-3.480969,0.662822
2,-3,16,0.256890,-4.014336,0.531960
3,7,11,0.462870,-3.516513,0.655810
4,-1,9,0.180915,-3.818800,0.501983
5,1,11,0.281587,-3.693661,0.535781
6,18,10,0.828235,-3.906192,0.485250
7,12,18,0.579914,-3.920768,0.412660


All the pareto_k values are less than 0.7, so despite the divergences, the model fit appears good and the LOO estimates are reliable. Looking at the loo_pit values, the $0^\text{th}$ observation (y = 28, $\sigma$ = 15) appears to be influential, as its PIT value is closer to 1. The same conclusion can also be drawn from the elpd values, where the 0th observation is more negative compared to the remaining seven observations.

Lets fit a new model by using StudentT distribution instead of normal for the likelihood. StudentT has heavier tails, it gives more probability for values far away from the mean than normal distribution. 

In [15]:
with pm.Model() as m2:
    μ = pm.Normal("μ", mu=0, sigma=50)
    τ = pm.HalfNormal("τ", sigma=10)
    nu = pm.Uniform("nu", 1, 10)
    θ = pm.Normal("theta",mu=μ,sigma=τ, shape=len(y))
    # studentT instead of normal
    pm.StudentT("likelihood",mu=θ,nu=nu, sigma=σ, observed=y)
    trace2 = pm.sample(draws=3000,tune=2000,target_accept=0.95, 
        return_inferencedata=True)
    pm.compute_log_likelihood(trace2)
    pm.sample_posterior_predictive(trace2, extend_inferencedata=True)

c:\ProgramData\miniconda3\envs\pymc_env\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
NUTS[nutpie]: [μ, τ, nu, theta]


Output()

Output()

Sampling: [likelihood]


Output()

The two models can be compared using PSIS-LOO-CV

In [16]:
arviz_stats.compare(
    {
        "M1 - Normal": arviz_stats.loo(trace),
        "M2 - StudentT": arviz_stats.loo(trace2)
    },
    round_to="none"
)

,rank,elpd,p,elpd_diff,weight,se,dse,warning
M1 - Normal,0,-30.954388,1.323621,0.000000,1.0,0.933937,0.000000,False
M2 - StudentT,1,-31.506613,1.327994,-0.552225,0.0,0.918935,0.049214,False


The first model with normal likelihood has a less negative elpd value and hence is better 

### References

1. Gelfand & Chang (1992). *Model determination using predictive distributions with implementation via sampling-based methods*.
https://statistics.stanford.edu/technical-reports/model-determination-using-predictive-distributions-implementation-sampling-based

2. Vehtari, Gelman, & Hwang (2013). *Understanding predictive information criteria for Bayesian models*. 
https://arxiv.org/abs/1307.5928

3. Vehtari, Gelman, & Gabry (2017). *Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC*.  
https://arxiv.org/abs/1507.04544

4. Vehtari, Gelman, Simpson, Yao, & Gabry (2024). *Pareto smoothed importance sampling*.  
https://arxiv.org/abs/1507.02646 

5. DB Rubin (1981). *Estimation in Parallel Randomized Experiments*.
https://www.jstor.org/stable/1164617